In [0]:
%python
dbutils.widgets.text("storage_account", "your_storage_account_name", "Storage Account Name")
dbutils.widgets.text("container", "olist", "Container Name")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")

adls_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net/olist"

print(f"Usando Storage Account: {storage_account}")
print(f"Usando Container: {container}")
print(f"Usando Root: {adls_root}")

In [0]:
DROP CATALOG IF EXISTS olist CASCADE;

In [0]:
%python
       

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS olist MANAGED LOCATION '{adls_root}/';
""")

spark.sql("USE CATALOG olist;")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS landing;
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS bronze MANAGED LOCATION '{adls_root}/bronze'
        COMMENT 'Camada Bronze — dados brutos ingeridos do CSV';
""")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS silver MANAGED LOCATION '{adls_root}/silver'
        COMMENT 'Camada Silver — dados limpos e deduplicados';
""")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS gold MANAGED LOCATION '{adls_root}/gold'
        COMMENT 'Camada Gold — modelagem dimensional';
""")

spark.sql("USE SCHEMA landing;")
spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS files LOCATION '{adls_root}/landing/files'
        COMMENT 'Arquivos brutos recebidos do dataset OLIST';
""")
spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS checkpoints LOCATION '{adls_root}/landing/checkpoints'
        COMMENT 'Checkpoints do Auto Loader (Bronze e Silver)';
""")
spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS schema LOCATION '{adls_root}/landing/schema'
        COMMENT 'Schema inference do Auto Loader (Bronze)';
""")

In [0]:
%python

tables = [
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers"
]

base_path = "/Volumes/olist/landing/files"

for table in tables:
    dbutils.fs.mkdirs(f"{base_path}/{table}/")